In [2]:
import os
import time
import pandas as pd
import pymysql
from selenium import webdriver
from selenium.webdriver.common.by import By

# =========================
# 1. 폴더 자동 생성 (CSV 저장용)
# =========================
os.makedirs("data", exist_ok=True)

# =========================
# 2. Selenium 설정
# =========================
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

driver.get("https://www.opinet.co.kr/user/cufaq/cufaqSelect.do")
time.sleep(3)

data_list = []

# =========================
# 3. 크롤링
# =========================
for page_num in range(1, 5):
    print(f"\n[{page_num}] 페이지 수집 중...")

    driver.execute_script(f"fn_egov_link_page({page_num})")
    time.sleep(2)

    for index in range(len(driver.find_elements(By.CSS_SELECTOR, "td.t_left.input a"))):

        try:
            titles = driver.find_elements(By.CSS_SELECTOR, "td.t_left.input a")
            target = titles[index]

            title_text = target.text

            target.click()
            time.sleep(1.5)

            content = driver.find_element(By.CLASS_NAME, "view_contents").text

            data_list.append({
                "page": page_num,
                "title": title_text,
                "content": content
            })

            driver.back()
            time.sleep(1.5)

        except Exception as e:
            print("에러:", e)
            try:
                driver.back()
                time.sleep(1.5)
            except:
                pass

driver.quit()

print("\n크롤링 완료:", len(data_list), "개")

# =========================
# 4. DataFrame 변환
# =========================
df = pd.DataFrame(data_list)

# =========================
# 5. CSV 저장 (핵심 수정 부분)
# =========================
csv_path = "data/faq_data.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV 저장 완료:", csv_path)

# =========================
# 6. MySQL 연결
# =========================
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="1234",
    db="ssj",   # 👈 반드시 본인 DB 이름으로 수정
    charset="utf8"
)

cursor = conn.cursor()

# =========================
# 7. 테이블 생성 (없으면 자동 생성)
# =========================
cursor.execute("""
CREATE TABLE IF NOT EXISTS faq_table (
    id INT AUTO_INCREMENT PRIMARY KEY,
    page INT,
    title TEXT,
    content TEXT
)
""")

# =========================
# 8. DB 저장
# =========================
sql = "INSERT INTO faq_table (page, title, content) VALUES (%s, %s, %s)"

for item in data_list:
    try:
        cursor.execute(sql, (
            item["page"],
            item["title"],
            item["content"]
        ))
    except Exception as e:
        print("DB 저장 실패:", e)

conn.commit()
conn.close()

print("DB 저장 완료!")


[1] 페이지 수집 중...

[2] 페이지 수집 중...

[3] 페이지 수집 중...

[4] 페이지 수집 중...

크롤링 완료: 31 개
CSV 저장 완료: data/faq_data.csv


OperationalError: (1049, "Unknown database 'ssj'")